# When Huzinaga needs a level shift

`1-basic-usage.ipynb` showed the three projectors agreeing to $10^{-13}$ Ha and treated
`huz_level_shift` as optional insurance. This notebook shows a system where plain `"huz"` is
**wrong by 5.6 Ha**, and where the level shift is the difference between a correct embedding and
a silently broken one.

## The mechanism, in one paragraph

The Huzinaga operator $O_\text{huz} = -(FP + P^\dagger F)$ turns the environment block of the Fock
matrix into $-F_{BB}$, so an environment level at $\varepsilon$ lands at $-\varepsilon$. That is a
*reflection about zero*, not a push upwards. It only moves the environment out of the way when
$\varepsilon < 0$. If an occupied environment orbital has $\varepsilon > 0$, the flip drives it
**down**, and once it falls below the fragment HOMO aufbau filling puts fragment electrons into
an environment orbital. The embedding is then not merely inaccurate, it is solving a different
problem.

The precise condition for trouble is

$$-\varepsilon_\text{max}^{B} \;<\; \varepsilon_\text{HOMO}^{A} \qquad\Longleftrightarrow\qquad \varepsilon_\text{max}^{B} + \varepsilon_\text{HOMO}^{A} > 0,$$

where $\varepsilon_\text{max}^{B}$ is the *most positive* eigenvalue of $F_{BB}$.

## Why anions

Occupied orbital energies are normally comfortably negative, which is why this almost never bites.
An anion is the exception: the extra electron raises every level, and in a basis with no diffuse
functions the excess density has nowhere to relax to, so the highest occupied orbitals are pushed
to *positive* energy. Put that anionic centre in the **environment** and Huzinaga reflects it
straight down through the fragment's occupied space.

Methoxide, $\text{CH}_3\text{O}^-$, in STO-3G is a clean example: three of its nine occupied
orbitals sit above zero. The fragment is the methyl carbon, so the environment owns the anionic
oxygen and its high-lying lone pairs — exactly the wrong way round for Huzinaga.

Note the orientation matters. Had the fragment been the *oxygen*, the anionic levels would belong
to the fragment, the environment would be ordinary and negative, and plain `"huz"` would be fine.

In [1]:
import nbed.backend
nbed.backend.set_backend(use_gpu=False)
if nbed.backend.USING_GPU:
    print("Running on GPU")
    import cupy as np
    from gpu4pyscf import dft, scf
    from pyscf import gto, lo, mcscf
else:
    print("Running on CPU")
    import numpy as np
    from pyscf import cc, ci, dft, gto, lo, mcscf, scf

np.set_printoptions(linewidth=110, suppress=True)

Running on CPU


In [2]:
from nbed.act_env_space import lowdin_populations, orbital_spread
from nbed.emb_scf import EmbedSCF

## 1. The system

Methoxide, built by deleting the hydroxyl hydrogen from the methanol geometry of notebook 1. The
carbon is atom 1, and it is the fragment; the oxygen (atom 0) and its lone pairs land in the
environment.

`STO-3G` is doing real work here rather than just being fast — a minimal basis cannot accommodate
the extra electron, which is what forces the occupied levels above zero. Section 5 repeats the run
in `6-31G` and `6-31G*` to show how the margin recovers.

In [3]:
geometry = [
    ("O", (-0.6582, -0.0067,  0.1730)),   # atom 0: the anionic centre -> environment
    ("C", ( 0.7031,  0.0083, -0.1305)),   # atom 1: the fragment
    ("H", ( 0.9877,  0.8943, -0.7114)),
    ("H", ( 1.0155, -0.8918, -0.6742)),
    ("H", ( 1.2001,  0.0363,  0.8431)),
]

basis_set      = "STO-3G"
xc             = "b3lyp"
charge         = -1          # the whole point
spin           = 0           # closed shell: this is not a Roothaan artefact
max_memory     = 10_000

active_atm_idx = [1]         # the methyl carbon
n_occ_active   = 2
n_vir_active   = 2
max_spread     = 3.0
mu_val         = 1e9

mol = gto.Mole(atom=geometry, basis=basis_set, charge=charge, spin=spin,
               max_memory=max_memory).build()
print(f"nao = {mol.nao}   nelec = {mol.nelec}")

nao = 13   nelec = (9, 9)


In [4]:
global_scf = dft.RKS(mol, xc=xc)
global_scf.kernel()
Sao = global_scf.get_ovlp()
assert global_scf.converged

eps_occ = global_scf.mo_energy[global_scf.mo_occ > 0]
n_pos = int((eps_occ > 0).sum())

print(f"\nglobal {xc.upper()} = {global_scf.e_tot:.10f}")
print(f"occupied orbital energies:\n{np.round(eps_occ, 4)}")
print(f"\n--> {n_pos} of {len(eps_occ)} occupied orbitals have POSITIVE energy.")
print("    In a neutral molecule this list would be entirely negative; those positive")
print("    levels are what Huzinaga will reflect downwards.")

converged SCF energy = -113.32750445216

global B3LYP = -113.3275044522
occupied orbital energies:
[-18.1584  -9.6631  -0.431   -0.2412  -0.0616  -0.0613   0.171    0.379    0.3842]

--> 3 of 9 occupied orbitals have POSITIVE energy.
    In a neutral molecule this list would be entirely negative; those positive
    levels are what Huzinaga will reflect downwards.


## 2. Partition, with the anionic centre in the environment

Same helpers as notebook 1, repeated here so this notebook stands alone.

In [ ]:
def localise_blocks(mol, mf):
    """Pipek-Mezey within each occupation block, leaving the total density untouched."""
    if nbed.backend.USING_GPU:
        mf = mf.to_cpu()
        mo_coeff = mf.mo_coeff.get()
        mo_occ = mf.mo_occ.get()
    else:
        mo_coeff = mf.mo_coeff
        mo_occ = mf.mo_occ
    
    C = mo_coeff.copy()
    for mask in (mo_occ > 1, mo_occ == 1, mo_occ == 0):
        if mask.sum() > 1:
            C[:, mask] = lo.PipekMezey(mol, mo_coeff[:, mask]).kernel()
    assert np.allclose(mf.make_rdm1(mo_coeff=C, mo_occ=mo_occ),
                       mf.make_rdm1(mo_coeff=mo_coeff, mo_occ=mo_occ), atol=1e-9)
    return np.asarray(C)


def pick_fragment(mol, mf, C, atom_idx, n_occ, n_vir, max_spread=None):
    """Rank occupied and virtual orbitals separately by target-atom population."""
    _, population = lowdin_populations(mol, C, atom_idx, drop_core_1s=True)
    eligible = np.ones_like(population, dtype=bool)
    if max_spread is not None:
        eligible &= orbital_spread(mol, C) <= max_spread
    occ_pool = np.where((mf.mo_occ > 0) & eligible)[0]
    vir_pool = np.where((mf.mo_occ == 0) & eligible)[0]
    pick_occ = np.sort(occ_pool[np.argsort(-population[occ_pool])[:n_occ]])
    pick_vir = np.sort(vir_pool[np.argsort(-population[vir_pool])[:n_vir]])
    active_idxs = np.concatenate([pick_occ, pick_vir])
    return active_idxs, np.setdiff1d(np.arange(mol.nao), active_idxs), population


def env_overlap(emb_obj, mf, Sao):
    """Largest overlap between an occupied environment orbital and an occupied embedded one."""
    C_env = emb_obj.C_full_reidx[:, emb_obj.env_idx_occ]
    return np.abs(C_env.conj().T @ Sao @ mf.mo_coeff[:, mf.mo_occ > 0]).max()


C_loc = localise_blocks(mol, global_scf)
active_idxs, env_idxs, population = pick_fragment(
    mol, global_scf, C_loc, active_atm_idx, n_occ_active, n_vir_active, max_spread
)

emb_obj = EmbedSCF(global_scf, active_idxs, env_idxs, C_loc, global_scf.mo_occ,
                   Sao, max_memory, mu_val=mu_val)

print(f"fragment MOs      : {active_idxs}  (C weight "
      f"{np.round(population[active_idxs], 2)})")
print(f"subsystem nelec   : {emb_obj.mol_act.nelec}")
print(f"occupied env cols : {emb_obj.env_idx_occ}")

fragment MOs      : [ 2  4  9 10]  (C weight [0.55 0.47 0.5  0.5 ])
subsystem nelec   : (2, 2)
occupied env cols : [0 1 2 3 4 5 6]


## 3. The prediction, before running anything

Diagonalise $F_{BB}$ using the *global* Fock matrix. Huzinaga will return each of these
eigenvalues with the opposite sign, so the flipped spectrum can be read off in advance. Whatever
is most positive here becomes the lowest environment level after embedding, and that is what has
to clear the fragment HOMO.

In [6]:
C_env = emb_obj.C_full_reidx[:, emb_obj.env_idx_occ]
eps_pre = np.linalg.eigvalsh(C_env.T @ global_scf.get_fock() @ C_env)

print(f"eps(F_BB) before the flip : {np.round(eps_pre, 4)}")
print(f"predicted after the flip  : {np.round(np.sort(-eps_pre), 4)}")
print(f"\nmost positive pre-flip eigenvalue = {eps_pre.max():+.4f}"
      f"  ->  lands at {-eps_pre.max():+.4f}")
print("The environment's lowest level will therefore be NEGATIVE, and it has to compete")
print("with the fragment's own occupied orbitals.")

eps(F_BB) before the flip : [-18.1578  -9.6626  -0.22    -0.1133  -0.0613   0.3731   0.379 ]
predicted after the flip  : [-0.379  -0.3731  0.0613  0.1133  0.22    9.6626 18.1578]

most positive pre-flip eigenvalue = +0.3790  ->  lands at -0.3790
The environment's lowest level will therefore be NEGATIVE, and it has to compete
with the fragment's own occupied orbitals.


## 4. All three projectors on the same partition

Watch three numbers: the energy error against the global DFT reference (which DFT-in-DFT must
reproduce exactly), the largest fragment–environment overlap (which must vanish), and the aufbau
margin (which must be positive).

In [7]:
results = {}
for label, kwargs in (
    ("mu",        dict(proj_type="mu")),
    ("huz",       dict(proj_type="huz")),
    ("huz+shift", dict(proj_type="huz", huz_level_shift=1e6)),
):
    e_tot, mf, emb_corr, env_cols, env_plus_corr = emb_obj.build_emb_dft(xc, warn=False, **kwargs)
    occ = mf.mo_occ > 0
    margin = mf.mo_energy[env_cols].min() - mf.mo_energy[occ].max()
    results[label] = dict(e_tot=e_tot, mf=mf, env_cols=env_cols, margin=margin,
                          err=e_tot - global_scf.e_tot, orth=env_overlap(emb_obj, mf, Sao))

print(f"{'projector':>10}  {'E error / Ha':>14}  {'max overlap':>12}  {'margin / Ha':>12}")
for label, r in results.items():
    print(f"{label:>10}  {r['err']:>+14.3e}  {r['orth']:>12.1e}  {r['margin']:>+12.4f}")

Overwritten attributes  get_hcore  of <class 'pyscf.dft.rks.RKS'>


SCF not converged.
SCF energy = 29.7127286645899


Overwritten attributes  get_fock get_hcore  of <class 'pyscf.dft.rks.RKS'>


converged SCF energy = 35.3134367545004
converged SCF energy = 29.7127286583901
 projector    E error / Ha   max overlap   margin / Ha
        mu      +6.611e-09       2.0e-10  +999999981.8750
       huz      +5.601e+00       1.0e+00       -0.1100
 huz+shift      -1.279e-13       2.0e-16  +999999.6538


`huz` is out by several hartree and the overlap is essentially 1 — meaning an environment orbital
is *fully* occupied in the fragment calculation. `mu` is fine, because parking the environment at
$+\mu$ does not care what sign $\varepsilon_\text{env}$ had. `huz+shift` is fine and, unlike `mu`,
exact.

The orbital listing makes the collapse unmistakable: for `huz` the occupied columns and the
environment columns are *the same columns*.

In [8]:
for label, r in results.items():
    mf = r["mf"]
    occ_cols = np.where(mf.mo_occ > 0)[0]
    shared = np.intersect1d(occ_cols, r["env_cols"])
    print(f"--- {label} ---")
    print(f"  occupied columns    : {occ_cols}   eps = {np.round(mf.mo_energy[occ_cols], 4)}")
    print(f"  environment columns : {r['env_cols']}")
    print(f"  occupied AND environment : {shared}"
          f"{'   <-- fragment electrons sitting in the environment' if len(shared) else '   (disjoint, as required)'}")
    print()

--- mu ---
  occupied columns    : [0 1]   eps = [-0.1866 -0.0328]
  environment columns : [ 6  7  8  9 10 11 12]
  occupied AND environment : []   (disjoint, as required)

--- huz ---
  occupied columns    : [0 1]   eps = [-1.7105 -1.6005]
  environment columns : [ 0  1  2  6  9 11 12]
  occupied AND environment : [0 1]   <-- fragment electrons sitting in the environment

--- huz+shift ---
  occupied columns    : [0 1]   eps = [-0.1867 -0.0328]
  environment columns : [ 6  7  8  9 10 11 12]
  occupied AND environment : []   (disjoint, as required)



`check_embedding` is designed to catch precisely this, so it should refuse the `huz` run. Wrapped
in `try` here so the notebook can keep going.

In [9]:
for label, r in results.items():
    mf = r["mf"]
    try:
        emb_obj.check_embedding(mf.mo_coeff, mf.mo_occ, mf.mo_energy, Sao, label)
        print(f"  ^^ {label}: PASSED\n")
    except AssertionError as err:
        print(f"  ^^ {label}: REJECTED -> {err}\n")

--- mu ---
  environment landed in columns : [ 6  7  8  9 10 11 12]
  eps(environment)              : [9.99999982e+08 9.99999990e+08 1.00000000e+09 1.00000000e+09 1.00000000e+09 1.00000000e+09 1.00000000e+09]
  occupied columns              : [0 1]
  eps(occupied)                 : [-0.1866 -0.0328]
  max |<env occ| S |emb occ>|   : 1.98e-10   <- must be ~0
  aufbau margin                 : +999999981.8750 Ha  <- must be > 0
  ^^ mu: PASSED

--- huz ---
  environment landed in columns : [ 0  1  2  6  9 11 12]
  eps(environment)              : [-1.7105 -1.6005 -1.4224  0.4425  0.6195 10.5696 11.2333]
  occupied columns              : [0 1]
  eps(occupied)                 : [-1.7105 -1.6005]
  max |<env occ| S |emb occ>|   : 9.98e-01   <- must be ~0
  aufbau margin                 : -0.1100 Ha  <- must be > 0
  ^^ huz: REJECTED -> occupied orbitals are contaminated by the environment

--- huz+shift ---
  environment landed in columns : [ 6  7  8  9 10 11 12]
  eps(environment)           

## 5. How large does $\lambda$ have to be?

Not large. The requirement is only that the shifted environment clears the fragment HOMO,

$$\lambda \;>\; \varepsilon_\text{max}^{B} + \varepsilon_\text{HOMO}^{A},$$

which here is a few tenths of a hartree. The $\lambda = 10^6$ used above is wild overkill; it is
convenient only because it also drives the environment to the last columns, which keeps orbital
windows well behaved. Note the energy is *identical* for every $\lambda$ that works — the shift
buys ordering, never accuracy.

In [10]:
homo_A = results["huz+shift"]["mf"].mo_energy[results["huz+shift"]["mf"].mo_occ > 0].max()
lam_needed = eps_pre.max() + homo_A
print(f"eps_max(B) = {eps_pre.max():+.4f},  eps_HOMO(A) = {homo_A:+.4f}"
      f"  ->  need lambda > {lam_needed:.4f} Ha\n")

print(f"{'lambda':>12}  {'E error / Ha':>14}  {'max overlap':>12}  {'margin / Ha':>12}   status")
for lam in (0.0, 0.1, 0.25, 0.5, 1.0, 10.0, 1e3, 1e6):
    e_tot, mf, _, env_cols, _ = emb_obj.build_emb_dft(xc, proj_type="huz",
                                                      huz_level_shift=lam,
                                                      warn=False)
    occ = mf.mo_occ > 0
    margin = mf.mo_energy[env_cols].min() - mf.mo_energy[occ].max()
    err, orth = e_tot - global_scf.e_tot, env_overlap(emb_obj, mf, Sao)
    ok = margin > 0 and orth < 1e-8 and abs(err) < 1e-8
    print(f"{lam:>12.2f}  {err:>+14.2e}  {orth:>12.1e}  {margin:>+12.4f}   "
          f"{'ok' if ok else 'BROKEN'}")

eps_max(B) = +0.3790,  eps_HOMO(A) = -0.0328  ->  need lambda > 0.3462 Ha

      lambda    E error / Ha   max overlap   margin / Ha   status


converged SCF energy = 35.3134367545004
        0.00       +5.60e+00       1.0e+00       -0.1100   BROKEN
converged SCF energy = 35.3134367545004
        0.10       +5.60e+00       1.0e+00       -0.1100   BROKEN
converged SCF energy = 35.3299565621558
        0.25       +5.62e+00       9.9e-01       -0.0433   BROKEN
converged SCF energy = 29.7127286583901
        0.50       -1.42e-13       3.4e-15       +0.1538   ok
converged SCF energy = 29.7127286583901
        1.00       -1.42e-13       1.5e-15       +0.6538   ok
converged SCF energy = 29.7127286583901
       10.00       -1.42e-13       4.8e-16       +9.6538   ok
converged SCF energy = 29.7127286583901
     1000.00       -1.42e-13       1.4e-16     +999.6538   ok
converged SCF energy = 29.7127286583901
  1000000.00       -1.28e-13       2.0e-16  +999999.6538   ok


## 6. When does this bite in practice?

Sweeping the basis shows how narrow the escape is. STO-3G pushes the environment to
$\varepsilon \approx +0.38$ and breaks for every fragment size. A split-valence basis brings it
down to $\approx +0.11$, which survives — but with only $0.03$–$0.11$ Ha of headroom, on a
quantity nobody normally checks.

The lesson is not "STO-3G is bad". It is that the margin is a property of the system, it is not
bounded below by anything, and plain `"huz"` gives no warning when it goes negative. Adding
$\lambda$ makes the margin something you choose instead of something you inherit.

In [11]:
def margin_for(basis, n_occ, huz_level_shift=0.0):
    m = gto.Mole(atom=geometry, basis=basis, charge=charge, spin=spin,
                 max_memory=max_memory, verbose=0).build()
    g = dft.RKS(m, xc=xc)
    g.verbose = 0
    g.kernel()
    S = g.get_ovlp()
    C = localise_blocks(m, g)
    act, env, _ = pick_fragment(m, g, C, active_atm_idx, n_occ, n_vir_active, max_spread)
    eo = EmbedSCF(g, act, env, C, g.mo_occ, S, max_memory, mu_val=mu_val)
    Cb = eo.C_full_reidx[:, eo.env_idx_occ]
    eps_max = np.linalg.eigvalsh(Cb.T @ g.get_fock() @ Cb).max()
    e, mf, _, ec, _ = eo.build_emb_dft(xc, proj_type="huz",
                                       huz_level_shift=huz_level_shift,
                                       warn=False)
    occ = mf.mo_occ > 0
    return (eps_max,
            mf.mo_energy[ec].min() - mf.mo_energy[occ].max(),
            e - g.e_tot,
            np.abs(Cb.T @ S @ mf.mo_coeff[:, occ]).max())


print(f"{'basis':>8} {'n_occ':>6} {'eps_max(B)':>11} {'margin':>9} {'E error':>10} "
      f"{'overlap':>9}   plain huz")
for basis in ("STO-3G", "6-31G", "6-31G*"):
    for n_occ in (1, 2, 3, 4):
        eps_max, margin, err, orth = margin_for(basis, n_occ)
        bad = margin < 0 or orth > 1e-8 or abs(err) > 1e-6
        print(f"{basis:>8} {n_occ:>6} {eps_max:>+11.4f} {margin:>+9.4f} {err:>+10.1e} "
              f"{orth:>9.1e}   {'BROKEN' if bad else 'ok'}")

   basis  n_occ  eps_max(B)    margin    E error   overlap   plain huz
  STO-3G      1     +0.3842   +0.0000   +2.1e+00   9.1e-01   BROKEN
  STO-3G      2     +0.3790   -0.1100   +5.6e+00   1.0e+00   BROKEN
  STO-3G      3     +0.3747   -0.1035   +1.1e+01   8.2e-01   BROKEN
  STO-3G      4     +0.3703   -1.1281   +9.6e+00   1.0e+00   BROKEN
   6-31G      1     +0.1132   +0.1136   -2.1e-12   8.9e-16   ok
   6-31G      2     +0.1065   +0.0428   -4.1e-12   2.3e-15   ok
   6-31G      3     +0.1001   +0.0491   -5.8e-12   2.6e-15   ok
   6-31G      4     +0.1001   +0.0481   -8.8e-11   1.7e-15   ok
  6-31G*      1     +0.1142   +0.1035   -7.2e-12   2.3e-15   ok
  6-31G*      2     +0.1078   +0.0338   -1.2e-11   3.9e-15   ok
  6-31G*      3     +0.1015   +0.0398   -1.4e-11   9.5e-15   ok
  6-31G*      4     +0.1015   +0.0388   -1.2e-10   5.4e-15   ok


## Takeaways

- Huzinaga *reflects* the environment about zero rather than pushing it up. That is only a shift
  out of the way when $\varepsilon_\text{env} < 0$.
- Occupied levels go positive in anions, especially in compact basis sets, and the failure needs
  nothing exotic beyond that — this example is closed-shell RKS, no open-shell Roothaan
  subtleties involved.
- Which subsystem holds the anionic centre decides everything. Anion in the **environment** is the
  dangerous orientation; anion in the fragment is harmless.
- When it fails it fails *silently and enormously*: 5.6 Ha here, with a converged SCF and no
  warning. Only the overlap and the aufbau margin reveal it, which is why `check_embedding`
  asserts on both.
- `huz_level_shift` needs to beat
  $\varepsilon_\text{max}^{B} + \varepsilon_\text{HOMO}^{A}$, typically a fraction of a hartree.
  A large $\lambda$ costs no accuracy and additionally parks the environment in the last columns.
- Practical default: pass `huz_level_shift` unless you have checked the margin. It cannot hurt the
  energy, and it converts a silent catastrophe into a non-event.